# The AI Telco Troubleshooting Challenge

Goal: Enhance the accuracy of Qwen2.5-1.5B when answering telco troubleshooting questions in telelogs data.

## I/ Set-up

In [ ]:
!git clone https://github.com/Kodro23/Telco-challenge

### 1. Load libraries

In [ ]:
import pandas as pd
import numpy as np
import re

### 2.Load data

In [ ]:
#load data
df = pd.read_csv("/content/Telco-challenge/train.csv")
print(df.head())

In [ ]:
#Function to clean questions
def clean_question(question):
    lines=question.split("\n")
    content=[]
    for line in lines:
        if "|" in line:
            content.append(line.split("|"))
    drive_test_data=[{"Observation": i+1,**dict(zip(content[0],row))} for i,row in enumerate(content[1:])]
    engineering_params=[{"Observation": i+1,**dict(zip(content[11],row))} for i,row in enumerate(content[12:])]
    #clean question
    ##recompute begining of the prompt
    q=""
    for l in [l+"\n" for l in lines[:19]]:
        q=q+l
    ##assemble
    cleaned_question= f" Question: {q} \nDrive test data: {drive_test_data} \nEngineering parameters {engineering_params} "
    return cleaned_question
#Apply to dataset
df["cleaned_question"]=df["question"].apply(lambda x: clean_question(x))
print(df["cleaned_question"][0])

## II/ Untrained model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
torch.cuda.empty_cache()

In [ ]:

# load the tokenizer and the model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="sequential",
    offload_folder="offload"
)

In [ ]:
def find_the_root_cause(question):
  messages = [
    {"role": "user", "content": question}
  ]
  text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
    )
  model_inputs = tokenizer([text], return_tensors="pt")
  model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
  # conduct text completion
  with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32
        )
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
  content = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

  match = re.search(r"\\boxed\{(C\d)\}", content)
  if match:
      return match.group(1)
  else:
      return "UNKNOWN"


In [ ]:
df["root_cause"]=df["cleaned_question"].apply(lambda x: find_the_root_cause(x))